# Day 08 — Circular
### #30DayChartChallenge | April 2026

**When does India get its rain?** A radial bar chart comparing average monthly rainfall across 4 IMD meteorological subdivisions with dramatically different monsoon patterns — from Assam & Meghalaya (heavy SW monsoon) to Rajasthan (arid desert).

**Data:** India Meteorological Department (IMD) — subdivision-wise actual rainfall, 1971–2017 averages  
**Source:** [Open Government Data Platform India](https://data.gov.in/catalog/rainfall-india) / [GitHub mirror](https://github.com/elakapoor/Rainfall-in-India)  
**Author:** Sharfudeen Yasar Arafath

In [ ]:
# — packages ------------------------------------------------------------------

library(ggplot2)
library(dplyr)
library(showtext)
library(sysfonts)
library(ggtext)

In [ ]:
# — fonts & display size ------------------------------------------------------

font_add_google("Outfit", "outfit")
font_add_google("Roboto Condensed", "roboto_condensed")
font_add_google("JetBrains Mono", "jetbrains")
showtext_auto()
showtext_opts(dpi = 300)

options(repr.plot.width = 14, repr.plot.height = 14, repr.plot.res = 300)

In [ ]:
# — read data -----------------------------------------------------------------
# Source: IMD subdivision-wise actual rainfall (1971-2017 averages)
# via data.gov.in / github.com/elakapoor/Rainfall-in-India

df <- read.csv("../../data/day_08/india_states_monthly_rainfall.csv",
               stringsAsFactors = FALSE)

# Use month_num (continuous) for x-axis — required for coord_polar + annotate
df$state <- factor(df$state,
  levels = c("Assam & Meghalaya", "Kerala", "Tamil Nadu", "Rajasthan")
)

# — annual totals (for caption) ------------------------------------------------
annual <- df %>%
  group_by(state) %>%
  summarise(total = round(sum(rainfall_mm)), .groups = "drop")

# — peak month per state (for selective labeling) ------------------------------
peaks <- df %>%
  group_by(state) %>%
  slice_max(rainfall_mm, n = 1) %>%
  ungroup()

# Pre-compute dodge positions for peak labels
# 4 groups, dodge width 0.75
state_offsets <- c("Assam & Meghalaya" = -0.28125, "Kerala" = -0.09375,
                   "Tamil Nadu" = 0.09375, "Rajasthan" = 0.28125)
peaks$x_dodge <- peaks$month_num + state_offsets[as.character(peaks$state)]

head(df)
annual
peaks

In [ ]:
# — theme & palette -----------------------------------------------------------

bg       <- "#0a0e17"
txt      <- "#E6EDF3"
txt_dim  <- "#8B949E"
txt_cap  <- "#484F58"
grid_col <- "#1a2030"

state_colors <- c(
  "Assam & Meghalaya" = "#A78BFA",
  "Kerala"            = "#34D399",
  "Tamil Nadu"        = "#FB923C",
  "Rajasthan"         = "#F472B6"
)

month_labels <- c("Jan", "Feb", "Mar", "Apr", "May", "Jun",
                  "Jul", "Aug", "Sep", "Oct", "Nov", "Dec")

In [ ]:
# — build the plot ------------------------------------------------------------

p <- ggplot(df, aes(x = month_num, y = rainfall_mm, fill = state)) +

  # ─── subtle monsoon season background wedges ───
  annotate("rect",
    xmin = 5.5, xmax = 9.5, ymin = 0, ymax = 660,
    fill = "#22553a", alpha = 0.15
  ) +
  annotate("rect",
    xmin = 9.5, xmax = 12.5, ymin = 0, ymax = 660,
    fill = "#552238", alpha = 0.15
  ) +

  # ─── concentric reference circles ───
  geom_hline(
    yintercept = c(100, 200, 300, 400, 500, 600),
    color = "#222e42", linewidth = 0.3, linetype = "dashed"
  ) +

  # ─── reference ring labels ───
  annotate("text",
    x = rep(0.6, 6), y = c(100, 200, 300, 400, 500, 600),
    label = c("100", "200", "300", "400", "500", "600 mm"),
    color = txt_dim, size = 2.4, family = "jetbrains",
    hjust = 0.5
  ) +

  # ─── main bars: grouped (dodged) per month ───
  geom_col(
    position = position_dodge(width = 0.75),
    width = 0.65, alpha = 0.9
  ) +

  # ─── thin white stroke on bars for glow effect ───
  geom_col(
    position = position_dodge(width = 0.75),
    width = 0.65,
    alpha = 0.12, fill = NA, color = "white", linewidth = 0.2
  ) +

  # ─── peak month labels (pre-computed dodge positions) ───
  geom_text(
    data = peaks,
    aes(x = x_dodge, y = rainfall_mm + 25,
        label = paste0(round(rainfall_mm), " mm")),
    inherit.aes = FALSE,
    color = txt, family = "jetbrains", size = 1.9,
    fontface = "bold"
  ) +

  # ─── monsoon season labels ───
  annotate("text",
    x = 7.5, y = 640,
    label = "SW MONSOON",
    color = "#4dbb6e", size = 2.5, family = "outfit",
    fontface = "bold", alpha = 0.65
  ) +
  annotate("text",
    x = 11, y = 640,
    label = "NE MONSOON",
    color = "#dd6688", size = 2.5, family = "outfit",
    fontface = "bold", alpha = 0.65
  ) +

  # ─── center annotation ───
  annotate("text",
    x = 6.5, y = -35,
    label = "INDIA\nMONSOON\nRAINFALL",
    color = txt_dim, size = 2.8, family = "outfit",
    fontface = "bold", lineheight = 0.85
  ) +

  # ─── scales ───
  scale_fill_manual(values = state_colors) +
  scale_x_continuous(
    breaks = 1:12, labels = month_labels,
    limits = c(0.5, 12.5), expand = c(0, 0)
  ) +
  scale_y_continuous(limits = c(-60, 700), expand = c(0, 0)) +

  # ─── polar coordinates ───
  coord_polar(start = 0) +

  # ─── labels ───
  labs(
    title    = "When Does India Get Its Rain?",
    subtitle = paste0(
      "Average monthly rainfall (mm) across 4 IMD subdivisions with contrasting monsoon patterns\n",
      "<span style='color:#4dbb6e;font-weight:bold'>Southwest monsoon</span> ",
      "(Jun\u2013Sep) drenches the northeast & Kerala, while the ",
      "<span style='color:#dd6688;font-weight:bold'>northeast monsoon</span> ",
      "(Oct\u2013Dec) feeds Tamil Nadu"
    ),
    x       = NULL,
    y       = NULL,
    fill    = NULL,
    caption = paste0(
      "Data: India Meteorological Department (IMD) \u2014 subdivision-wise actual rainfall, 1971\u20132017 averages\n",
      "Source: data.gov.in / github.com/elakapoor/Rainfall-in-India (IMD open data)\n",
      "Annual totals \u2014 Assam & Meghalaya: 2,492 mm \u00b7 Kerala: 2,767 mm \u00b7 ",
      "Tamil Nadu: 920 mm \u00b7 Rajasthan: 478 mm\n",
      "#30DayChartChallenge \u00b7 Day 08: Circular \u00b7 ",
      "Sharfudeen Yasar Arafath"
    )
  ) +

  # ─── theme ───
  theme_minimal(base_family = "roboto_condensed") +
  theme(
    plot.title = element_text(
      family = "outfit", face = "bold", size = 26,
      hjust = 0.5, color = txt,
      margin = margin(t = 10, b = 8)
    ),
    plot.subtitle = element_markdown(
      size = 11, hjust = 0.5, color = txt_dim,
      lineheight = 1.4,
      margin = margin(b = 15)
    ),
    plot.caption = element_text(
      size = 7.5, hjust = 0.5, color = txt_cap,
      lineheight = 1.5,
      margin = margin(t = 15, b = 10)
    ),

    legend.position = "bottom",
    legend.direction = "horizontal",
    legend.text = element_text(size = 11, color = txt, family = "outfit"),
    legend.key.size = unit(0.5, "cm"),
    legend.key = element_rect(fill = bg, color = NA),
    legend.spacing.x = unit(0.3, "cm"),
    legend.margin = margin(t = -5, b = 5),

    axis.text.x = element_text(
      size = 11, color = txt, face = "bold",
      family = "outfit"
    ),
    axis.text.y = element_blank(),

    panel.grid.major.x = element_line(color = grid_col, linewidth = 0.2),
    panel.grid.major.y = element_blank(),
    panel.grid.minor = element_blank(),

    plot.background  = element_rect(fill = bg, color = NA),
    panel.background = element_rect(fill = bg, color = NA),
    plot.margin      = margin(15, 30, 15, 30)
  )

p

In [ ]:
# — save ----------------------------------------------------------------------

ggsave("../../chart/day_08_circular.png",
       plot = p, width = 14, height = 14, dpi = 300, bg = bg)

cat("Done \u2014 saved to chart/day_08_circular.png\n")